# 🚢 Pipeline de Tratamento de Dados — Importações Brasileiras
## Portfólio · Analista de Dados Júnior

**Objetivo:** Ingestão, validação, limpeza, engenharia de features e exportação  
dos dados de importação para consumo no Power BI.

**Stack:** Python 3.11 · Pandas · NumPy  
**Período:** 2023–2024  
**Fonte:** Dataset sintético gerado localmente (modelo Data Warehouse)

---

### 📐 Estrutura do Pipeline

| Etapa | Descrição |
|-------|-----------|
| **01** | Configuração do ambiente e constantes |
| **02** | Ingestão e inspeção inicial |
| **03** | Validação de integridade relacional (FKs) |
| **04** | Tipagem correta e padronização |
| **05** | Limpeza e tratamento de qualidade |
| **06** | Filtro de período 2023–2024 |
| **07** | Engenharia de features |
| **08** | Agregações analíticas |
| **09** | Exportação para Power BI |

## ⚙️ Etapa 01 — Configuração do Ambiente

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

# ── Caminhos ──────────────────────────────────────────────────────────────────
RAW_DIR    = Path("dados_brutos")          # CSVs originais
OUTPUT_DIR = Path("dados_tratados")        # CSVs exportados para Power BI
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Constantes de negócio ─────────────────────────────────────────────────────
PERIODO_INICIO = "2023-01-01"
PERIODO_FIM    = "2024-12-31"

TIPOS_CUSTO_TRIBUTARIOS = ["Imposto Importação", "ICMS"]
TIPOS_CUSTO_LOGISTICOS  = ["Frete Internacional", "Seguro", "Armazenagem",
                            "Demurrage", "Despachante"]

CAMBIO_ALERTA = 5.50   # Limiar para sinalizar impacto cambial alto

print("✅ Ambiente configurado.")
print(f"   Output dir: {OUTPUT_DIR.resolve()}")

✅ Ambiente configurado.
   Output dir: C:\Users\luis-\Luis\ALURA\projeto analise de dados\Projeto Comex\Analise Dados Comex\dados_tratados


## 📥 Etapa 02 — Ingestão e Inspeção Inicial

In [4]:
def load_csv(filename: str, parse_dates: list = None) -> pd.DataFrame:
    """Carrega CSV com encoding UTF-8-sig (padrão exportado pelo pipeline gerador)."""
    path = RAW_DIR / filename
    df = pd.read_csv(path, encoding="utf-8-sig",
                     parse_dates=parse_dates if parse_dates else False)
    print(f"  ✔ {filename:<40} {df.shape[0]:>6} linhas × {df.shape[1]} colunas")
    return df

print("Carregando tabelas…\n")

dim_clientes     = load_csv("dim_clientes.csv")
dim_fornecedores = load_csv("dim_fornecedores.csv")
dim_produtos     = load_csv("dim_produtos.csv")
dim_tempo        = load_csv("dim_tempo.csv",    parse_dates=["data"])
fato_processos   = load_csv("fato_processos_importacao.csv")
fato_itens       = load_csv("fato_itens_processo.csv")
fato_custos      = load_csv("fato_custos.csv")

print("\n📊 Carga concluída.")

Carregando tabelas…

  ✔ dim_clientes.csv                            200 linhas × 5 colunas
  ✔ dim_fornecedores.csv                        100 linhas × 5 colunas
  ✔ dim_produtos.csv                            500 linhas × 6 colunas
  ✔ dim_tempo.csv                              1096 linhas × 5 colunas
  ✔ fato_processos_importacao.csv              1000 linhas × 10 colunas
  ✔ fato_itens_processo.csv                    2951 linhas × 5 colunas
  ✔ fato_custos.csv                            6174 linhas × 4 colunas

📊 Carga concluída.


In [5]:
def inspecionar(df: pd.DataFrame, nome: str) -> None:
    """Relatório rápido de qualidade: shape, tipos, nulos e duplicatas."""
    total      = len(df)
    nulos      = df.isnull().sum()
    nulos_pct  = (nulos / total * 100).round(2)
    dupl       = df.duplicated().sum()

    print(f"\n{'─'*55}")
    print(f"  {nome.upper()}")
    print(f"{'─'*55}")
    print(f"  Shape      : {df.shape}")
    print(f"  Duplicatas : {dupl}")
    print(f"  Nulos totais: {nulos.sum()}")
    if nulos.sum() > 0:
        print("  Nulos por coluna:")
        print(nulos[nulos > 0].to_frame("qtd").assign(pct=nulos_pct).to_string())
    print(f"  Tipos:\n{df.dtypes.to_string()}")

for nome, df in [
    ("dim_clientes",     dim_clientes),
    ("dim_fornecedores", dim_fornecedores),
    ("dim_produtos",     dim_produtos),
    ("fato_processos",   fato_processos),
    ("fato_itens",       fato_itens),
    ("fato_custos",      fato_custos),
]:
    inspecionar(df, nome)


───────────────────────────────────────────────────────
  DIM_CLIENTES
───────────────────────────────────────────────────────
  Shape      : (200, 5)
  Duplicatas : 0
  Nulos totais: 0
  Tipos:
id_cliente       int64
nome_empresa       str
segmento           str
estado             str
data_cadastro      str

───────────────────────────────────────────────────────
  DIM_FORNECEDORES
───────────────────────────────────────────────────────
  Shape      : (100, 5)
  Duplicatas : 0
  Nulos totais: 0
  Tipos:
id_fornecedor           int64
nome_empresa              str
pais                      str
lead_time_medio_dias    int64
confiabilidade            str

───────────────────────────────────────────────────────
  DIM_PRODUTOS
───────────────────────────────────────────────────────
  Shape      : (500, 6)
  Duplicatas : 0
  Nulos totais: 0
  Tipos:
id_produto              int64
descricao                 str
ncm                     int64
categoria                 str
peso_unitario_kg      f

In [6]:
fato_processos.head()

,id_processo,id_cliente,id_fornecedor,incoterm,modal,data_embarque,data_chegada_prevista,data_chegada_real,status,taxa_cambio_usd_brl
0,1,103,52,CIF,Marítimo,2023-04-08,2023-05-22,2023-05-20,Entregue,5.01
1,2,75,75,CIF,Aéreo,2022-07-11,2022-07-22,2022-07-27,Atrasado,4.98
2,3,88,100,CIF,Marítimo,2022-06-19,2022-08-14,2022-08-13,Entregue,4.97
3,4,53,2,EXW,Marítimo,2022-07-15,2022-09-11,2022-09-11,Entregue,4.84
4,5,88,30,EXW,Aéreo,2024-08-31,2024-09-09,2024-09-08,Desembaraçado,5.46


In [7]:
# Tratamento das colunas com datas

col_data = ['data_embarque', 'data_chegada_prevista', 'data_chegada_real']
fato_processos[col_data] = fato_processos[col_data].apply(pd.to_datetime, errors='coerce')

## 🔗 Etapa 03 — Validação de Integridade Relacional

In [8]:
def validar_fk(df_filho: pd.DataFrame, col_fk: str,
               df_pai: pd.DataFrame,   col_pk: str,
               nome: str) -> pd.Series:
    """
    Retorna os valores órfãos (presentes no filho mas ausentes no pai).
    Levanta ValueError se houver violações — garante que o pipeline falhe ruidosamente.
    """
    orphans = df_filho[col_fk][~df_filho[col_fk].isin(df_pai[col_pk])]
    if len(orphans) > 0:
        raise ValueError(
            f"❌ FK inválida: {nome}.{col_fk} → {len(orphans)} valores órfãos.\n"
            f"   Amostra: {orphans.unique()[:5].tolist()}"
        )
    print(f"  ✔ {nome:<40} {col_fk:<25} OK  ({len(df_filho):,} registros)")
    return orphans

print("Validando chaves estrangeiras…\n")

validar_fk(fato_processos, "id_cliente",    dim_clientes,     "id_cliente",    "fato_processos")
validar_fk(fato_processos, "id_fornecedor", dim_fornecedores, "id_fornecedor", "fato_processos")
validar_fk(fato_itens,     "id_processo",   fato_processos,   "id_processo",   "fato_itens")
validar_fk(fato_itens,     "id_produto",    dim_produtos,     "id_produto",    "fato_itens")
validar_fk(fato_custos,    "id_processo",   fato_processos,   "id_processo",   "fato_custos")

print("\n✅ Integridade relacional 100% validada.")

Validando chaves estrangeiras…

  ✔ fato_processos                           id_cliente                OK  (1,000 registros)
  ✔ fato_processos                           id_fornecedor             OK  (1,000 registros)
  ✔ fato_itens                               id_processo               OK  (2,951 registros)
  ✔ fato_itens                               id_produto                OK  (2,951 registros)
  ✔ fato_custos                              id_processo               OK  (6,174 registros)

✅ Integridade relacional 100% validada.


In [9]:
# ── Validação de unicidade das PKs ────────────────────────────────────────────
def validar_pk(df: pd.DataFrame, col_pk: str, nome: str) -> None:
    dupl = df[col_pk].duplicated().sum()
    if dupl > 0:
        raise ValueError(f"❌ PK duplicada: {nome}.{col_pk} — {dupl} duplicatas.")
    print(f"  ✔ {nome:<40} {col_pk:<25} OK")

print("Validando chaves primárias…\n")

validar_pk(dim_clientes,     "id_cliente",    "dim_clientes")
validar_pk(dim_fornecedores, "id_fornecedor", "dim_fornecedores")
validar_pk(dim_produtos,     "id_produto",    "dim_produtos")
validar_pk(fato_processos,   "id_processo",   "fato_processos")
validar_pk(fato_itens,       "id_item",       "fato_itens")
validar_pk(fato_custos,      "id_custo",      "fato_custos")

print("\n✅ Todas as PKs são únicas.")

Validando chaves primárias…

  ✔ dim_clientes                             id_cliente                OK
  ✔ dim_fornecedores                         id_fornecedor             OK
  ✔ dim_produtos                             id_produto                OK
  ✔ fato_processos                           id_processo               OK
  ✔ fato_itens                               id_item                   OK
  ✔ fato_custos                              id_custo                  OK

✅ Todas as PKs são únicas.


## 🏷️ Etapa 04 — Tipagem Correta e Padronização

In [10]:
# ── dim_clientes ──────────────────────────────────────────────────────────────
dim_clientes["data_cadastro"] = pd.to_datetime(dim_clientes["data_cadastro"])
dim_clientes["segmento"]      = dim_clientes["segmento"].astype("category")
dim_clientes["estado"]        = dim_clientes["estado"].astype("category")

# ── dim_fornecedores ──────────────────────────────────────────────────────────
dim_fornecedores["confiabilidade"] = pd.Categorical(
    dim_fornecedores["confiabilidade"],
    categories=["Baixa", "Média", "Alta"],
    ordered=True
)
dim_fornecedores["pais"] = dim_fornecedores["pais"].astype("category")

# ── dim_produtos ──────────────────────────────────────────────────────────────
# NCM deve ser string de 8 dígitos — zero-padding obrigatório
dim_produtos["ncm"]       = dim_produtos["ncm"].astype(str).str.zfill(8)
dim_produtos["categoria"] = dim_produtos["categoria"].astype("category")

# ── fato_processos ────────────────────────────────────────────────────────────
date_cols = ["data_embarque", "data_chegada_prevista", "data_chegada_real"]
for col in date_cols:
    fato_processos[col] = pd.to_datetime(fato_processos[col])

fato_processos["incoterm"] = pd.Categorical(
    fato_processos["incoterm"],
    categories=["FOB", "CIF", "EXW"],
    ordered=False
)
fato_processos["modal"]  = fato_processos["modal"].astype("category")
fato_processos["status"] = fato_processos["status"].astype("category")

print("✅ Tipagem aplicada em todas as tabelas.")
print("\nResumo de categorias:")
print(f"  Segmentos  : {dim_clientes['segmento'].cat.categories.tolist()}")
print(f"  Confiab.   : {dim_fornecedores['confiabilidade'].cat.categories.tolist()}")
print(f"  Incoterms  : {fato_processos['incoterm'].cat.categories.tolist()}")
print(f"  Modais     : {fato_processos['modal'].cat.categories.tolist()}")
print(f"  Status     : {fato_processos['status'].cat.categories.tolist()}")

✅ Tipagem aplicada em todas as tabelas.

Resumo de categorias:
  Segmentos  : ['Automotivo', 'Eletrônico', 'Industrial', 'Varejo']
  Confiab.   : ['Baixa', 'Média', 'Alta']
  Incoterms  : ['FOB', 'CIF', 'EXW']
  Modais     : ['Aéreo', 'Marítimo']
  Status     : ['Atrasado', 'Desembaraçado', 'Em trânsito', 'Entregue']


## 🧹 Etapa 05 — Limpeza e Qualidade dos Dados

In [11]:
# ── 5.1 Validação lógica das datas ────────────────────────────────────────────
mask_data_invalida = (
    fato_processos["data_chegada_prevista"] <= fato_processos["data_embarque"]
)
qtd_invalidas = mask_data_invalida.sum()
print(f"Registros com data_chegada_prevista <= data_embarque: {qtd_invalidas}")

if qtd_invalidas > 0:
    print("  → Descartando registros inválidos.")
    fato_processos = fato_processos[~mask_data_invalida].copy()

# ── 5.2 Verificar valores negativos em colunas métricas ───────────────────────
for df_name, df, col in [
    ("fato_itens",   fato_itens,   "valor_total_usd"),
    ("fato_itens",   fato_itens,   "quantidade"),
    ("fato_custos",  fato_custos,  "valor_brl"),
]:
    neg = (df[col] < 0).sum()
    print(f"  Negativos em {df_name}.{col}: {neg}")

# ── 5.3 Padronizar strings (strip + title case nos nomes) ─────────────────────
dim_clientes["nome_empresa"]     = dim_clientes["nome_empresa"].str.strip()
dim_fornecedores["nome_empresa"] = dim_fornecedores["nome_empresa"].str.strip()
dim_produtos["descricao"]        = dim_produtos["descricao"].str.strip()
fato_custos["tipo_custo"]        = fato_custos["tipo_custo"].str.strip()

# ── 5.4 Checar valores de câmbio fora do intervalo esperado ───────────────────
cambio_fora = fato_processos[
    (fato_processos["taxa_cambio_usd_brl"] < 4.80) |
    (fato_processos["taxa_cambio_usd_brl"] > 5.80)
]
print(f"\nProcessos com câmbio fora do intervalo [4.80, 5.80]: {len(cambio_fora)}")

print("\n✅ Limpeza concluída. Dataset íntegro.")

Registros com data_chegada_prevista <= data_embarque: 0
  Negativos em fato_itens.valor_total_usd: 0
  Negativos em fato_itens.quantidade: 0
  Negativos em fato_custos.valor_brl: 0

Processos com câmbio fora do intervalo [4.80, 5.80]: 0

✅ Limpeza concluída. Dataset íntegro.


## 📅 Etapa 06 — Filtro de Período 2023–2024

In [12]:
# Aplica filtro temporal no fato principal pela data de embarque
mask_periodo = (
    (fato_processos["data_embarque"] >= PERIODO_INICIO) &
    (fato_processos["data_embarque"] <= PERIODO_FIM)
)

fato_processos_23_24 = fato_processos[mask_periodo].copy()

# Propaga o filtro para as tabelas fato filhas
processos_validos = fato_processos_23_24["id_processo"]
fato_itens_23_24  = fato_itens[fato_itens["id_processo"].isin(processos_validos)].copy()
fato_custos_23_24 = fato_custos[fato_custos["id_processo"].isin(processos_validos)].copy()

print("Registros após filtro 2023–2024:\n")
print(f"  fato_processos : {len(fato_processos):>5}  →  {len(fato_processos_23_24):>5}")
print(f"  fato_itens     : {len(fato_itens):>5}  →  {len(fato_itens_23_24):>5}")
print(f"  fato_custos    : {len(fato_custos):>5}  →  {len(fato_custos_23_24):>5}")

# Distribuição por ano
print("\nDistribuição por ano (data_embarque):")
print(
    fato_processos_23_24["data_embarque"]
    .dt.year.value_counts()
    .sort_index()
    .to_string()
)

Registros após filtro 2023–2024:

  fato_processos :  1000  →    657
  fato_itens     :  2951  →   1903
  fato_custos    :  6174  →   4063

Distribuição por ano (data_embarque):
data_embarque
2023    313
2024    344


## 🔧 Etapa 07 — Engenharia de Features

Criação de colunas derivadas que enriquecem a análise e alimentam medidas no Power BI.

In [13]:
# ─── 7.1 Features temporais e de lead time ───────────────────────────────────
fp = fato_processos_23_24.copy()

fp["lead_time_previsto_dias"] = (
    fp["data_chegada_prevista"] - fp["data_embarque"]
).dt.days

fp["lead_time_real_dias"] = (
    fp["data_chegada_real"] - fp["data_embarque"]
).dt.days

fp["desvio_lead_time_dias"] = (
    fp["data_chegada_real"] - fp["data_chegada_prevista"]
).dt.days  # positivo = atrasado, negativo = adiantado

fp["flag_atraso"]      = (fp["desvio_lead_time_dias"] > 0).astype(int)
fp["flag_atraso_grave"] = (fp["desvio_lead_time_dias"] > 10).astype(int)

# Ano e mês do embarque (facilita slicers no Power BI)
fp["ano_embarque"] = fp["data_embarque"].dt.year
fp["mes_embarque"] = fp["data_embarque"].dt.month
fp["trimestre"]    = fp["data_embarque"].dt.quarter
fp["ano_mes"]      = fp["data_embarque"].dt.to_period("M").astype(str)

print("Lead time — estatísticas descritivas:")
print(fp[["lead_time_previsto_dias","lead_time_real_dias","desvio_lead_time_dias"]].describe().round(2))

Lead time — estatísticas descritivas:
       lead_time_previsto_dias  lead_time_real_dias  desvio_lead_time_dias
count                   657.00               657.00                 657.00
mean                     34.44                35.61                   1.17
std                      16.57                17.58                   5.21
min                       5.00                 3.00                  -2.00
25%                      25.00                25.00                  -2.00
50%                      36.00                37.00                  -1.00
75%                      48.00                49.00                   0.00
max                      60.00                79.00                  20.00


In [14]:
# ─── 7.2 Custo total e decomposição por processo ──────────────────────────────
custo_total = (
    fato_custos_23_24
    .groupby("id_processo")["valor_brl"]
    .sum()
    .rename("custo_total_brl")
)

custo_por_tipo = (
    fato_custos_23_24
    .pivot_table(index="id_processo", columns="tipo_custo",
                 values="valor_brl", aggfunc="sum", fill_value=0)
    .rename(columns=lambda c: f"custo_{c.lower().replace(' ', '_')}_brl")
)

fp = (
    fp
    .join(custo_total,    on="id_processo")
    .join(custo_por_tipo, on="id_processo")
)

# ─── 7.3 Valor total importado (USD e BRL) por processo ──────────────────────
valor_usd_processo = (
    fato_itens_23_24
    .groupby("id_processo")["valor_total_usd"]
    .sum()
    .rename("valor_total_usd")
)

fp = fp.join(valor_usd_processo, on="id_processo")
fp["valor_total_brl"]   = fp["valor_total_usd"] * fp["taxa_cambio_usd_brl"]
fp["custo_sobre_valor"] = (
    fp["custo_total_brl"] / fp["valor_total_brl"]
).round(4)   # % de custo logístico + tributário sobre valor da mercadoria

# ─── 7.4 Flag de impacto cambial alto ────────────────────────────────────────
fp["flag_cambio_alto"] = (fp["taxa_cambio_usd_brl"] >= CAMBIO_ALERTA).astype(int)

# ─── 7.5 Custo por kg (necessita peso total dos itens) ────────────────────────
peso_processo = (
    fato_itens_23_24
    .merge(dim_produtos[["id_produto", "peso_unitario_kg"]], on="id_produto")
    .assign(peso_total_kg=lambda d: d["quantidade"] * d["peso_unitario_kg"])
    .groupby("id_processo")["peso_total_kg"]
    .sum()
    .rename("peso_total_kg")
)
fp = fp.join(peso_processo, on="id_processo")
fp["custo_por_kg_brl"] = (fp["custo_total_brl"] / fp["peso_total_kg"]).round(2)

print("Features criadas:")
novas = ["lead_time_previsto_dias","lead_time_real_dias","desvio_lead_time_dias",
         "flag_atraso","flag_atraso_grave","custo_total_brl","valor_total_usd",
         "valor_total_brl","custo_sobre_valor","flag_cambio_alto",
         "peso_total_kg","custo_por_kg_brl","ano_embarque","mes_embarque",
         "trimestre","ano_mes"]
for col in novas:
    print(f"  + {col}")

Features criadas:
  + lead_time_previsto_dias
  + lead_time_real_dias
  + desvio_lead_time_dias
  + flag_atraso
  + flag_atraso_grave
  + custo_total_brl
  + valor_total_usd
  + valor_total_brl
  + custo_sobre_valor
  + flag_cambio_alto
  + peso_total_kg
  + custo_por_kg_brl
  + ano_embarque
  + mes_embarque
  + trimestre
  + ano_mes


In [15]:
# ─── 7.6 Enriquecer com dimensões (desnormalizado para análise) ───────────────
fp_enriquecido = (
    fp
    .merge(dim_clientes[["id_cliente","nome_empresa","segmento","estado"]],
           on="id_cliente", suffixes=("","_cliente"))
    .merge(dim_fornecedores[["id_fornecedor","nome_empresa","pais",
                              "lead_time_medio_dias","confiabilidade"]],
           on="id_fornecedor", suffixes=("","_fornecedor"))
)
fp_enriquecido.rename(columns={
    "nome_empresa"   : "nome_cliente",
    "nome_empresa_fornecedor": "nome_fornecedor",
}, inplace=True)

print(f"✅ fato_processos enriquecido: {fp_enriquecido.shape}")
print(f"   Colunas totais: {fp_enriquecido.columns.tolist()}")

✅ fato_processos enriquecido: (657, 40)
   Colunas totais: ['id_processo', 'id_cliente', 'id_fornecedor', 'incoterm', 'modal', 'data_embarque', 'data_chegada_prevista', 'data_chegada_real', 'status', 'taxa_cambio_usd_brl', 'lead_time_previsto_dias', 'lead_time_real_dias', 'desvio_lead_time_dias', 'flag_atraso', 'flag_atraso_grave', 'ano_embarque', 'mes_embarque', 'trimestre', 'ano_mes', 'custo_total_brl', 'custo_armazenagem_brl', 'custo_demurrage_brl', 'custo_despachante_brl', 'custo_frete_internacional_brl', 'custo_icms_brl', 'custo_imposto_importação_brl', 'custo_seguro_brl', 'valor_total_usd', 'valor_total_brl', 'custo_sobre_valor', 'flag_cambio_alto', 'peso_total_kg', 'custo_por_kg_brl', 'nome_cliente', 'segmento', 'estado', 'nome_fornecedor', 'pais', 'lead_time_medio_dias', 'confiabilidade']


## 📊 Etapa 08 — Agregações Analíticas

Tabelas sumarizadas que respondem diretamente às **10 perguntas de negócio**
e servem como base para os visuais do Power BI.

In [16]:
# ── 8.1 Lead time médio por fornecedor ───────────────────────────────────────
agg_lead_time_fornecedor = (
    fp_enriquecido
    .groupby(["id_fornecedor","nome_fornecedor","pais","confiabilidade"])
    .agg(
        total_processos      = ("id_processo",             "count"),
        lead_time_real_medio = ("lead_time_real_dias",     "mean"),
        lead_time_prev_medio = ("lead_time_previsto_dias", "mean"),
        desvio_medio_dias    = ("desvio_lead_time_dias",   "mean"),
        taxa_atraso_pct      = ("flag_atraso",             "mean"),
        processos_atrasados  = ("flag_atraso",             "sum"),
    )
    .round(2)
    .reset_index()
    .sort_values("lead_time_real_medio")
)

agg_lead_time_fornecedor["taxa_atraso_pct"] = (
    agg_lead_time_fornecedor["taxa_atraso_pct"] * 100
).round(1)

print("Top 10 fornecedores com menor lead time real:")
print(agg_lead_time_fornecedor.head(10)[
    ["nome_fornecedor","pais","confiabilidade",
     "lead_time_real_medio","taxa_atraso_pct","total_processos"]
].to_string(index=False))

Top 10 fornecedores com menor lead time real:
      nome_fornecedor          pais confiabilidade  lead_time_real_medio  taxa_atraso_pct  total_processos
  Methode Electronics         China          Média                 19.33             0.00                3
     Aptiv Components         Índia           Alta                 20.17             0.00                6
 Sensata Technologies        Itália           Alta                 22.20             0.00                5
    Yanmar Components           EUA           Alta                 23.00             0.00                1
  Brose Fahrzeugteile Coreia do Sul           Alta                 23.33             0.00                3
   Ferrari Industrial         China           Alta                 23.67             0.00                6
     Midea Industrial      Alemanha          Baixa                 24.00            25.00                8
 Mercedes Supply GmbH           EUA          Média                 24.14            29.00         

In [17]:
# ── 8.2 Impacto cambial nos custos ───────────────────────────────────────────
agg_cambio = (
    fp_enriquecido
    .assign(faixa_cambio=pd.cut(
        fp_enriquecido["taxa_cambio_usd_brl"],
        bins=[4.79, 5.00, 5.20, 5.40, 5.60, 5.81],
        labels=["4,80–5,00","5,01–5,20","5,21–5,40","5,41–5,60","5,61–5,80"]
    ))
    .groupby("faixa_cambio", observed=True)
    .agg(
        qtd_processos      = ("id_processo",       "count"),
        custo_medio_brl    = ("custo_total_brl",   "mean"),
        custo_total_brl    = ("custo_total_brl",   "sum"),
        valor_medio_usd    = ("valor_total_usd",   "mean"),
    )
    .round(2)
    .reset_index()
)

print("Impacto cambial por faixa USD/BRL:")
print(agg_cambio.to_string(index=False))

Impacto cambial por faixa USD/BRL:
faixa_cambio  qtd_processos  custo_medio_brl  custo_total_brl  valor_medio_usd
   4,80–5,00             17       956,811.09    16,265,788.45       491,502.04
   5,01–5,20            142       662,230.84    94,036,779.58       321,529.39
   5,21–5,40            243       704,171.06   171,113,568.17       315,382.79
   5,41–5,60            172       944,356.23   162,429,272.34       406,920.21
   5,61–5,80             83     1,176,894.33    97,682,229.12       441,597.42


In [18]:
# ── 8.3 Ranking de eficiência de fornecedores (score composto) ────────────────
def min_max_norm(s: pd.Series) -> pd.Series:
    """Normaliza para [0, 1]. Score = 1 é melhor."""
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

ranking = agg_lead_time_fornecedor.copy()

# Normaliza invertido para lead time e taxa de atraso (menor = melhor)
ranking["score_lead_time"] = 1 - min_max_norm(ranking["lead_time_real_medio"])
ranking["score_pontual"]   = 1 - min_max_norm(ranking["taxa_atraso_pct"])

# Custo de frete por fornecedor (join com fp_enriquecido)
custo_frete_forn = (
    fp_enriquecido
    .join(
        fato_custos_23_24[fato_custos_23_24["tipo_custo"] == "Frete Internacional"]
        .groupby("id_processo")["valor_brl"].sum()
        .rename("frete_brl"),
        on="id_processo"
    )
    .groupby("id_fornecedor")["frete_brl"]
    .mean()
    .rename("frete_medio_brl")
)
ranking = ranking.join(custo_frete_forn, on="id_fornecedor")
ranking["score_frete"] = 1 - min_max_norm(ranking["frete_medio_brl"])

# Score final ponderado
ranking["score_eficiencia"] = (
    ranking["score_lead_time"] * 0.40 +
    ranking["score_pontual"]   * 0.40 +
    ranking["score_frete"]     * 0.20
).round(4)

ranking_final = (
    ranking
    .sort_values("score_eficiencia", ascending=False)
    .reset_index(drop=True)
    .assign(posicao=lambda d: d.index + 1)
)

print("🏆 Top 10 fornecedores mais eficientes:")
print(ranking_final.head(10)[
    ["posicao","nome_fornecedor","pais","confiabilidade",
     "lead_time_real_medio","taxa_atraso_pct","score_eficiencia"]
].to_string(index=False))

🏆 Top 10 fornecedores mais eficientes:
 posicao        nome_fornecedor          pais confiabilidade  lead_time_real_medio  taxa_atraso_pct  score_eficiencia
       1      Yanmar Components           EUA           Alta                 23.00             0.00              0.95
       2   Sensata Technologies        Itália           Alta                 22.20             0.00              0.95
       3       Aptiv Components         Índia           Alta                 20.17             0.00              0.95
       4    Brose Fahrzeugteile Coreia do Sul           Alta                 23.33             0.00              0.95
       5     Ferrari Industrial         China           Alta                 23.67             0.00              0.94
       6    Methode Electronics         China          Média                 19.33             0.00              0.91
       7  BASF Chemical Germany         Japão           Alta                 25.60             0.00              0.90
       8     Gent

In [19]:
# ── 8.4 Custo logístico por tipo ao longo do tempo ───────────────────────────
agg_custo_tempo = (
    fato_custos_23_24
    .merge(fp_enriquecido[["id_processo","ano_embarque","mes_embarque",
                            "trimestre","ano_mes","modal","incoterm"]],
           on="id_processo")
    .groupby(["ano_mes","tipo_custo"])["valor_brl"]
    .sum()
    .reset_index()
    .rename(columns={"valor_brl": "custo_total_brl"})
    .sort_values(["ano_mes","tipo_custo"])
)

print("Custo por tipo (últimos 6 meses no dataset):")
ultimos_meses = agg_custo_tempo["ano_mes"].unique()[-6:]
print(
    agg_custo_tempo[agg_custo_tempo["ano_mes"].isin(ultimos_meses)]
    .pivot_table(index="ano_mes", columns="tipo_custo",
                 values="custo_total_brl", aggfunc="sum")
    .round(0)
    .to_string()
)

Custo por tipo (últimos 6 meses no dataset):
tipo_custo  Armazenagem  Demurrage  Despachante  Frete Internacional          ICMS  Imposto Importação       Seguro
ano_mes                                                                                                            
2024-06       97,251.00  46,740.00   121,369.00         8,716,993.00 15,826,792.00       13,258,961.00   699,731.00
2024-07       63,067.00  16,302.00    81,386.00         9,490,033.00  9,055,434.00        9,337,348.00   421,024.00
2024-08       84,536.00  29,207.00    91,656.00        10,297,919.00 10,599,299.00        9,184,720.00   657,914.00
2024-09      112,657.00  16,435.00   125,575.00        18,130,012.00 21,451,886.00       18,957,244.00 1,316,129.00
2024-10       86,055.00   2,150.00   120,519.00        15,372,862.00 10,713,217.00       10,883,715.00   562,815.00
2024-11       77,189.00  56,105.00    73,102.00         6,558,963.00  4,916,389.00        5,254,562.00   259,506.00


In [20]:
# ── 8.5 Performance por segmento de cliente ──────────────────────────────────
agg_segmento = (
    fp_enriquecido
    .groupby("segmento")
    .agg(
        total_processos      = ("id_processo",        "count"),
        valor_total_usd      = ("valor_total_usd",    "sum"),
        valor_total_brl      = ("valor_total_brl",    "sum"),
        custo_total_brl      = ("custo_total_brl",    "sum"),
        custo_medio_processo = ("custo_total_brl",    "mean"),
        lead_time_medio      = ("lead_time_real_dias","mean"),
        taxa_atraso_pct      = ("flag_atraso",        "mean"),
    )
    .round(2)
    .reset_index()
    .sort_values("valor_total_brl", ascending=False)
)
agg_segmento["taxa_atraso_pct"] *= 100
agg_segmento["share_valor_pct"] = (
    agg_segmento["valor_total_brl"] / agg_segmento["valor_total_brl"].sum() * 100
).round(1)

print("Performance por segmento de cliente:")
print(agg_segmento.to_string(index=False))

Performance por segmento de cliente:
  segmento  total_processos  valor_total_usd  valor_total_brl  custo_total_brl  custo_medio_processo  lead_time_medio  taxa_atraso_pct  share_valor_pct
Industrial              177    70,498,367.51   379,032,283.92   161,158,608.73            910,500.61            34.28            18.00            29.80
    Varejo              197    66,781,521.51   358,988,405.76   149,233,371.29            757,529.80            35.95            22.00            28.20
Eletrônico              148    52,053,494.24   281,565,237.92   126,350,577.77            853,720.12            34.88            20.00            22.10
Automotivo              135    47,960,203.84   254,078,844.55   104,785,079.87            776,185.78            37.68            21.00            19.90


In [21]:
# ── 8.6 Custo do atraso (Demurrage + Armazenagem nos atrasados) ──────────────
custo_atraso = (
    fato_custos_23_24[
        fato_custos_23_24["tipo_custo"].isin(["Demurrage","Armazenagem"])
    ]
    .merge(fp_enriquecido[["id_processo","flag_atraso","trimestre",
                            "ano_embarque","desvio_lead_time_dias"]],
           on="id_processo")
)

agg_custo_atraso = (
    custo_atraso
    .groupby(["ano_embarque","trimestre","tipo_custo"])["valor_brl"]
    .agg(["sum","count","mean"])
    .rename(columns={"sum":"total_brl","count":"qtd","mean":"medio_brl"})
    .round(2)
    .reset_index()
)

custo_oportunidade = custo_atraso[custo_atraso["flag_atraso"] == 1]["valor_brl"].sum()
print(f"💸 Custo de oportunidade dos processos atrasados (Demurrage + Armazenagem):")
print(f"   R$ {custo_oportunidade:,.2f}")
print("\nDetalhe trimestral:")
print(agg_custo_atraso.to_string(index=False))

💸 Custo de oportunidade dos processos atrasados (Demurrage + Armazenagem):
   R$ 453,750.62

Detalhe trimestral:
 ano_embarque  trimestre  tipo_custo  total_brl  qtd  medio_brl
         2023          1 Armazenagem 221,090.05   85   2,601.06
         2023          1   Demurrage  63,801.38   13   4,907.80
         2023          2 Armazenagem 239,173.41   85   2,813.80
         2023          2   Demurrage  50,953.86   12   4,246.16
         2023          3 Armazenagem 210,287.94   84   2,503.43
         2023          3   Demurrage  66,836.41   16   4,177.28
         2023          4 Armazenagem 156,943.45   59   2,660.06
         2023          4   Demurrage  41,241.02    7   5,891.57
         2024          1 Armazenagem 247,932.80   95   2,609.82
         2024          1   Demurrage  75,640.02   17   4,449.41
         2024          2 Armazenagem 248,166.22   94   2,640.07
         2024          2   Demurrage 116,539.42   25   4,661.58
         2024          3 Armazenagem 260,260.55   94   

## 💾 Etapa 09 — Exportação para Power BI

Todos os CSVs exportados com:
- Encoding `utf-8-sig` (compatível com Excel/Power BI sem caracteres especiais)
- Separador vírgula
- Datas no formato `YYYY-MM-DD`

In [22]:
def exportar(df: pd.DataFrame, nome: str, subdir: str = "") -> None:
    """Exporta DataFrame como CSV tratado, pronto para Power BI."""
    dest = OUTPUT_DIR / subdir
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f"{nome}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
    print(f"  ✔ {nome:<45} {len(df):>6} linhas  →  {path}")

print("Exportando tabelas dimensão…\n")
exportar(dim_clientes,                    "dim_clientes")
exportar(dim_fornecedores,                "dim_fornecedores")
exportar(dim_produtos,                    "dim_produtos")
exportar(dim_tempo[
    dim_tempo["data"].dt.year.isin([2023,2024])
],                                        "dim_tempo")

print("\nExportando tabelas fato (tratadas, período 2023–2024)…\n")
exportar(fp_enriquecido,                  "fato_processos_importacao")
exportar(fato_itens_23_24,                "fato_itens_processo")
exportar(fato_custos_23_24,               "fato_custos")

print("\nExportando agregações analíticas…\n")
exportar(agg_lead_time_fornecedor,        "agg_lead_time_por_fornecedor",  "agregacoes")
exportar(agg_cambio,                      "agg_impacto_cambial",           "agregacoes")
exportar(ranking_final,                   "agg_ranking_fornecedores",      "agregacoes")
exportar(agg_custo_tempo,                 "agg_custo_por_tipo_tempo",      "agregacoes")
exportar(agg_segmento,                    "agg_performance_segmento",      "agregacoes")
exportar(agg_custo_atraso,                "agg_custo_atraso",              "agregacoes")

print("\n✅ Exportação concluída!")
print(f"📁 Diretório: {OUTPUT_DIR.resolve()}")

Exportando tabelas dimensão…

  ✔ dim_clientes                                     200 linhas  →  dados_tratados\dim_clientes.csv
  ✔ dim_fornecedores                                 100 linhas  →  dados_tratados\dim_fornecedores.csv
  ✔ dim_produtos                                     500 linhas  →  dados_tratados\dim_produtos.csv
  ✔ dim_tempo                                        731 linhas  →  dados_tratados\dim_tempo.csv

Exportando tabelas fato (tratadas, período 2023–2024)…

  ✔ fato_processos_importacao                        657 linhas  →  dados_tratados\fato_processos_importacao.csv
  ✔ fato_itens_processo                             1903 linhas  →  dados_tratados\fato_itens_processo.csv
  ✔ fato_custos                                     4063 linhas  →  dados_tratados\fato_custos.csv

Exportando agregações analíticas…

  ✔ agg_lead_time_por_fornecedor                     100 linhas  →  dados_tratados\agregacoes\agg_lead_time_por_fornecedor.csv
  ✔ agg_impacto_cambial       

---

## 📋 Resumo do Pipeline

| Etapa | Descrição | Status |
|-------|-----------|--------|
| 01 | Configuração do ambiente | ✅ |
| 02 | Ingestão e inspeção inicial | ✅ |
| 03 | Validação de integridade relacional | ✅ |
| 04 | Tipagem correta e padronização | ✅ |
| 05 | Limpeza e qualidade dos dados | ✅ |
| 06 | Filtro de período 2023–2024 | ✅ |
| 07 | Engenharia de features | ✅ |
| 08 | Agregações analíticas | ✅ |
| 09 | Exportação para Power BI | ✅ |

### 🆕 Features geradas (`fato_processos_importacao.csv`)

| Coluna | Descrição |
|--------|-----------|
| `lead_time_previsto_dias` | Dias entre embarque e chegada prevista |
| `lead_time_real_dias` | Dias entre embarque e chegada real |
| `desvio_lead_time_dias` | Desvio real vs previsto (positivo = atraso) |
| `flag_atraso` | 1 se processo atrasou, 0 se não |
| `flag_atraso_grave` | 1 se atraso > 10 dias |
| `flag_cambio_alto` | 1 se câmbio ≥ 5,50 BRL/USD |
| `custo_total_brl` | Soma de todos os custos do processo em BRL |
| `valor_total_usd` | Valor total dos itens importados em USD |
| `valor_total_brl` | Valor total convertido para BRL |
| `custo_sobre_valor` | Ratio custo logístico / valor mercadoria |
| `peso_total_kg` | Peso total dos itens do processo |
| `custo_por_kg_brl` | Custo logístico por kg |
| `ano_mes` | Período no formato YYYY-MM (slicer no Power BI) |

### 📦 Arquivos exportados para Power BI

```
dados_tratados/
├── dim_clientes.csv
├── dim_fornecedores.csv
├── dim_produtos.csv
├── dim_tempo.csv
├── fato_processos_importacao.csv   ← principal, com todas as features
├── fato_itens_processo.csv
├── fato_custos.csv
└── agregacoes/
    ├── agg_lead_time_por_fornecedor.csv
    ├── agg_impacto_cambial.csv
    ├── agg_ranking_fornecedores.csv
    ├── agg_custo_por_tipo_tempo.csv
    ├── agg_performance_segmento.csv
    └── agg_custo_atraso.csv
```